In [33]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [34]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))
from config import raw_data, control_data, external_data, raw_kyiv, filtered_data, daily_data

#### text

In [35]:
df_text = pd.read_csv(filtered_data / "text_messages_filtered.csv")

df_text["datetime"] = pd.to_datetime(df_text["datetime"])
df_text["date"] = df_text["datetime"].dt.date
df_text["hour"] = df_text["datetime"].dt.hour
df_text["weekday"] = df_text["datetime"].dt.weekday 

df_text.head(2)

,donation_id,id,conversation_id,sender_id,datetime,word_count,donor_sender_id,is_donor,date,hour,weekday
0,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,e39038a3-9136-4e60-bd59-7405cdaa8953,01842182-4154-4143-b10d-32c21307f4fd,50475d11-75e7-4d96-a0e5-5c62d7f67c3a,2022-05-22 22:47:59.730,1,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2022-05-22,22,6
1,06b923ca-4d2d-44bb-8b20-4e2c9b08ce9f,7157c4a6-8137-4c4e-849d-03628c89a001,01842182-4154-4143-b10d-32c21307f4fd,50475d11-75e7-4d96-a0e5-5c62d7f67c3a,2022-05-22 22:48:10.917,5,b5518648-ab76-42f8-8a2b-f539f9d9cf47,0,2022-05-22,22,6


In [36]:
donor_messages = df_text[df_text["is_donor"] == 1]

word count


In [37]:
daily_wordcount = (donor_messages
        .groupby(["donation_id", "date"])["word_count"]
        .sum()
        .reset_index()
        .rename(columns={"word_count": "donor_daily_word_count"})
)

message count

In [38]:
daily_msgcount = (donor_messages
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_message_count")
)

active chats

In [39]:
daily_active_chats = (df_text
        .groupby(["donation_id", "date"])["conversation_id"]
        .nunique()
        .reset_index()
        .rename(columns={"conversation_id": "daily_active_chats"})
)

active hours

In [40]:
daily_active_hours = (donor_messages
        .groupby(["donation_id", "date"])["hour"]
        .nunique()
        .reset_index()
        .rename(columns={"hour": "donor_daily_active_hours"})
)

average length

In [41]:
daily_avg_length = (
    donor_messages
    .groupby(["donation_id", "date"])["word_count"]
    .mean()
    .reset_index()
    .rename(columns={"word_count": "donor_daily_avg_length"})
)

time shares of text messages 

In [42]:
donor_messages["is_night"] = ((donor_messages["hour"] >= 0) & 
                              (donor_messages["hour"] < 6)).astype(int)

donor_messages["is_morning"] = ((donor_messages["hour"] >= 6) & 
                                (donor_messages["hour"] < 12)).astype(int)

donor_messages["is_afternoon"] = ((donor_messages["hour"] >= 12) & 
                                  (donor_messages["hour"] < 18)).astype(int)

donor_messages["is_evening"] = ((donor_messages["hour"] >= 18) & 
                                (donor_messages["hour"] < 24)).astype(int)


daily_time_shares = (
    donor_messages
    .groupby(["donation_id", "date"])[
        ["is_morning", "is_afternoon", "is_evening", "is_night"]
    ]
    .mean()
    .reset_index()
    .rename(columns={
        "is_morning": "morning_share",
        "is_afternoon": "afternoon_share",
        "is_evening": "evening_share",
        "is_night": "night_share"
    })
)

In [43]:
received_messages = df_text[df_text['is_donor'] == 0]

In [44]:
chat_to_donor = (
    df_text[df_text['is_donor'] == 1][['donation_id', 'conversation_id']]
    .drop_duplicates()
    .rename(columns={'donation_id': 'donor_id'}) 
)
received_with_donor = received_messages.merge(
    chat_to_donor,
    on='conversation_id',
    how='inner'
)

received messages

In [45]:
n_messages_received = (
    received_with_donor
    .groupby(['donor_id', 'date'])
    .size()
    .reset_index(name='n_messages_received')
    .rename(columns={'donor_id': 'donation_id'}) 
)

received words

In [46]:
words_received = (
    received_with_donor
    .groupby(['donor_id', 'date'])['word_count']
    .sum()
    .reset_index()
    .rename(columns={'donor_id': 'donation_id', 'word_count': 'words_received'})
)

frac of sent words to top5 chats

In [47]:
top5_chats = (
    donor_messages
    .groupby(['donation_id', 'conversation_id'])['word_count']
    .sum()
    .reset_index()
    .sort_values(['donation_id', 'word_count'], ascending=[True, False])
    .groupby('donation_id')
    .head(5)
)[['donation_id', 'conversation_id']]

words_in_top5 = (
    donor_messages
    .merge(top5_chats, on=['donation_id', 'conversation_id'], how='inner')
    .groupby(['donation_id', 'date'])['word_count']
    .sum()
    .reset_index()
    .rename(columns={'word_count': 'words_in_top5'})
)

words_total = (
    donor_messages
    .groupby(['donation_id', 'date'])['word_count']
    .sum()
    .reset_index()
    .rename(columns={'word_count': 'words_total'})
)

frac_words_top5 = words_total.merge(
    words_in_top5, on=['donation_id', 'date'], how='left'
).fillna(0)

frac_words_top5['frac_words_closest_5_contacts'] = (
    frac_words_top5['words_in_top5'] /
    (frac_words_top5['words_total'] + 1e-9)
)

frac_words_closest_5 = frac_words_top5[[
    'donation_id', 'date', 'frac_words_closest_5_contacts'
]]


#### comments

In [48]:
comments_path = filtered_data / "comments_sorted.csv"

df_comments = pd.read_csv(comments_path)

df_comments["datetime"] = pd.to_datetime(df_comments["datetime"])
df_comments["date"] = df_comments["datetime"].dt.date

comment count

In [49]:
daily_comment_count = (
    df_comments
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_comment_count")
)

average comment length

In [50]:
daily_comment_avg_length = (
    df_comments
        .groupby(["donation_id", "date"])["word_count"]
        .mean()
        .reset_index()
        .rename(columns={"word_count": "donor_daily_comment_avg_length"})
)

In [51]:
daily_comments = daily_comment_count.merge(
    daily_comment_avg_length,
    on=["donation_id", "date"],
    how="left"
)

#### reactions

In [52]:
reactions_path = filtered_data / "reactions_sorted.csv"

df_reactions = pd.read_csv(reactions_path)
df_reactions["datetime"] = pd.to_datetime(df_reactions["datetime"])
df_reactions["date"] = df_reactions["datetime"].dt.date

reaction count

In [53]:
daily_reactions = (
    df_reactions
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_reaction_count")
)

night share reactions

In [54]:
df_reactions['hour'] = pd.to_datetime(df_reactions['datetime']).dt.hour
df_reactions['is_night'] = ((df_reactions['hour'] >= 0) &
                            (df_reactions['hour'] < 6)).astype(int)

night_share_reactions = (
    df_reactions
    .groupby(['donation_id', 'date'])['is_night']
    .mean()
    .reset_index()
    .rename(columns={'is_night': 'night_share_reactions'})
)

#### posts 

In [55]:
posts_path = filtered_data / "posts_sorted.csv"

df_posts = pd.read_csv(posts_path)

df_posts["datetime"] = pd.to_datetime(df_posts["datetime"])
df_posts["date"] = df_posts["datetime"].dt.date

posts count

In [56]:
daily_post_count = (
    df_posts
        .groupby(["donation_id", "date"])
        .size()
        .reset_index(name="donor_daily_post_count")
)

average length of the text in posts

In [57]:
daily_post_avg_length = (
    df_posts
        .groupby(["donation_id", "date"])["word_count"]
        .mean()
        .reset_index()
        .rename(columns={"word_count": "donor_daily_post_avg_length"})
)

total media 

In [58]:
daily_post_total_media = (
    df_posts
        .groupby(["donation_id", "date"])["media_count"]
        .sum()
        .reset_index()
        .rename(columns={"media_count": "donor_daily_post_total_media"})
)

In [59]:
daily_posts = (
    daily_post_count
        .merge(daily_post_avg_length, on=["donation_id", "date"], how="left")
        .merge(daily_post_total_media, on=["donation_id", "date"], how="left")
)

#### audio

In [60]:
df_audio = pd.read_csv(filtered_data / "audio_messages_filtered.csv")

df_audio["datetime"] = pd.to_datetime(df_audio["datetime"])
df_audio["date"] = df_audio["datetime"].dt.date

df_audio_donor = df_audio[df_audio["is_donor"] == 1]


audio count

In [61]:
daily_audio_count = (
    df_audio_donor
    .groupby(["donation_id", "date"])
    .size()
    .reset_index(name="donor_daily_audio_count")
)

average length audio

In [62]:
audio_avg_length = (
    df_audio_donor
    .groupby(["donation_id", "date"])
    .agg(
        donor_daily_audio_count=("length_seconds", "count"),
        donor_daily_audio_avg_length=("length_seconds", "mean"),
    )
    .reset_index()
)

In [63]:
save_base = daily_data

files_to_save = {
    'donor_daily_word_count.csv': daily_wordcount,
    'donor_daily_message_count.csv': daily_msgcount,
    'daily_active_chats.csv': daily_active_chats,
    'donor_daily_active_hours.csv': daily_active_hours,
    'donor_daily_avg_length.csv': daily_avg_length,
    'donor_daily_time_shares.csv': daily_time_shares,
    'donor_daily_messages_received.csv': n_messages_received,
    'donor_daily_words_received.csv': words_received,
    'donor_daily_frac_words_top5.csv': frac_words_closest_5,
    'donor_daily_comments.csv': daily_comments,
    'donor_daily_reactions.csv': daily_reactions,
    'donor_daily_night_share_reactions.csv': night_share_reactions,
    'donor_daily_posts.csv': daily_posts,
    'donor_daily_audio.csv': daily_audio_count,
    'donor_daily_audio_avg_length.csv': audio_avg_length
    
}

for filename, df in files_to_save.items():
    df.to_csv(str(save_base / filename), index=False)